In [ ]:


import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import os

# 1. 데이터 로드 및 결측치 처리
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

train = train.drop_duplicates(subset=[col for col in train.columns if col != 'ID']).reset_index(drop=True)
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

# 2. 파생변수 19개 생성
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']
    return data

train = add_features(train)
test = add_features(test)

# 3. 인코딩 및 스케일링 (딥러닝 필수 과정)
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}
train['activity'] = train['activity'].map(activity_map)
test['activity'] = test['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
test['edu_level'] = test['edu_level'].map(edu_map)

nominal_cols = ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']
for feature in nominal_cols:
    le = LabelEncoder()
    le = le.fit(train[feature])
    train[feature] = le.transform(train[feature])
    unseen = [label for label in np.unique(test[feature]) if label not in le.classes_]
    if unseen: le.classes_ = np.append(le.classes_, unseen)
    test[feature] = le.transform(test[feature])

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score']
x_test = test.drop('ID', axis=1)

# MLP는 거리/가중치 기반이므로 모든 특성의 스케일을 통일해야 합니다.
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# 4. 딥러닝(MLP) 모델 학습 및 5-Fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(x_train))
test_pred = np.zeros(len(x_test))

# 은닉층 3개(128, 64, 32), 조기 종료 적용, L2 규제(alpha) 설정
for tr_idx, va_idx in kf.split(x_train_scaled):
    X_tr, X_va = x_train_scaled[tr_idx], x_train_scaled[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
    
    mlp = MLPRegressor(
        hidden_layer_sizes=(128, 64, 32), 
        activation='relu', 
        solver='adam', 
        alpha=0.01, 
        batch_size=32, 
        learning_rate_init=0.001, 
        max_iter=1000, 
        early_stopping=True, 
        validation_fraction=0.1, 
        n_iter_no_change=20, 
        random_state=42
    )
    
    mlp.fit(X_tr, y_tr)
    oof[va_idx] = mlp.predict(X_va)
    test_pred += mlp.predict(x_test_scaled) / kf.n_splits

print(f"딥러닝(MLP) 최종 CV MAE: {mean_absolute_error(y_train, oof):.4f}")

# 5. 제출 파일 저장
os.makedirs('../submissions', exist_ok=True)
test_pred = np.clip(test_pred, 0, 1)
sample_submission['stress_score'] = test_pred
submit_path = '../submissions/submit_08_dl_mlp.csv'
sample_submission.to_csv(submit_path, index=False)
print(f"★ 제출 파일 생성 완료: {submit_path}")

딥러닝(MLP) 최종 CV MAE: 0.2379
★ 제출 파일 생성 완료: ../submissions/submit_08_dl_mlp.csv
